### Problem Statement

A large e-commerce company has accumulated millions of customer transactions over several years.

The business team faces several challenges:

1. All customers are treated similarly in marketing campaigns.
1. Discount offers are sent to every customer, causing unnecessary revenue loss.
1. Premium customers are not receiving personalized services.
1. Customer retention strategies are ineffective because customer behavior patterns are unknown.
1. Marketing budget is increasing without proportional growth in sales.

The company wants to automatically group customers based on purchasing behavior so that targeted business strategies can be created for each group.

In [1]:
import os
import joblib
import warnings

import pandas as pd
import numpy as np

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

In [2]:
DATA_PATH = "./data/customer_segmentation_dataset.csv"

ARTIFACT_DIR = "./project-k_means_clustering"

os.makedirs(
    ARTIFACT_DIR,
    exist_ok=True
)

In [3]:
df = pd.read_csv(DATA_PATH)

print(f"Dataset Shape: {df.shape}")
df.head()

Dataset Shape: (50000, 10)


,Customer_ID,Age,Annual_Income,Total_Orders,Avg_Order_Value,Days_Since_Last_Purchase,App_Sessions_Per_Month,Website_Visits_Per_Month,Wishlist_Count,Cart_Additions_Per_Month
0,C100000,56,2194584,102,8188,324,138,166,46,11
1,C100001,69,425424,19,2549,330,64,121,94,47
2,C100002,46,410964,247,12034,23,13,3,6,57
3,C100003,32,152754,100,10569,213,99,59,55,97
4,C100004,60,1150896,227,9296,180,92,195,82,5


In [4]:
feature_columns = [
    "Age",
    "Annual_Income",
    "Total_Orders",
    "Avg_Order_Value",
    "Days_Since_Last_Purchase",
    "App_Sessions_Per_Month",
    "Website_Visits_Per_Month",
    "Wishlist_Count",
    "Cart_Additions_Per_Month"
]

X = df[feature_columns]

In [5]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

In [6]:
# Elbow Method
wcss = []

for k in range(2, 11):

    model = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=20
    )

    model.fit(X_scaled)

    wcss.append(model.inertia_)

    print(
        f"K={k} | WCSS={model.inertia_:.2f}"
    )

K=2 | WCSS=412417.80
K=3 | WCSS=389452.58
K=4 | WCSS=371084.21
K=5 | WCSS=357218.69
K=6 | WCSS=345511.07
K=7 | WCSS=334836.33
K=8 | WCSS=325393.67
K=9 | WCSS=317082.68
K=10 | WCSS=309595.66


In [7]:
OPTIMAL_K = 5

print(
    f"\nTraining Final KMeans Model (K={OPTIMAL_K})..."
)

kmeans = KMeans(
    n_clusters=OPTIMAL_K,
    random_state=42,
    n_init=20,
    max_iter=500
)

clusters = kmeans.fit_predict(
    X_scaled
)

df["Cluster"] = clusters


Training Final KMeans Model (K=5)...


In [8]:
# Silhouette Score
score = silhouette_score(
    X_scaled,
    clusters
)

print(
    f"\nSilhouette Score: {score:.4f}"
)


Silhouette Score: 0.0728


In [9]:
cluster_summary = (
    df
    .groupby("Cluster")
    .agg({
        "Age": "mean",
        "Annual_Income": "mean",
        "Total_Orders": "mean",
        "Avg_Order_Value": "mean",
        "Days_Since_Last_Purchase": "mean",
        "App_Sessions_Per_Month": "mean",
        "Website_Visits_Per_Month": "mean",
        "Wishlist_Count": "mean",
        "Cart_Additions_Per_Month": "mean",
        "Customer_ID": "count"
    })
)

cluster_summary.rename(
    columns={
        "Customer_ID": "Customer_Count"
    },
    inplace=True
)

print("\nCluster Summary\n")
print(cluster_summary)


Cluster Summary

               Age  Annual_Income  Total_Orders  Avg_Order_Value  \
Cluster                                                            
0        43.698102   2.216726e+06    178.867932      7795.677100   
1        44.342438   7.910510e+05    110.467136      7749.045459   
2        43.055499   2.237902e+06     61.699755      7891.744542   
3        43.870301   1.361121e+06    179.537850      7467.580438   
4        42.555643   1.347751e+06     96.431706      7367.517098   

         Days_Since_Last_Purchase  App_Sessions_Per_Month  \
Cluster                                                     
0                      189.797658               49.226474   
1                      174.852910               36.179133   
2                      183.346154               71.765150   
3                      178.770555              107.124503   
4                      182.433485              112.928444   

         Website_Visits_Per_Month  Wishlist_Count  Cart_Additions_Per_Month  

In [10]:
joblib.dump(
    kmeans,
    os.path.join(
        ARTIFACT_DIR,
        "kmeans_model.pkl"
    )
)

joblib.dump(
    scaler,
    os.path.join(
        ARTIFACT_DIR,
        "scaler.pkl"
    )
)

joblib.dump(
    feature_columns,
    os.path.join(
        ARTIFACT_DIR,
        "feature_columns.pkl"
    )
)

cluster_summary.to_csv(
    os.path.join(
        ARTIFACT_DIR,
        "cluster_summary.csv"
    )
)

df.to_csv(
    os.path.join(
        ARTIFACT_DIR,
        "clustered_customers.csv"
    ),
    index=False
)